# i2Nav V2 Backward-Derivative LOSO Sensitivity

This notebook retrains both the V1 covariance stage and V2 correction stage for 10 folds x 3 seeds after replacing centered acceleration derivatives with backward-only derivatives. It tests whether the learned model and later ledger survive removal of the known one-sample derivative lookahead. It is a **new sensitivity pipeline**, not the original frozen V2 and not proof of fully causal sensor-to-service operation: interpolation availability and execution/transport latency remain separate.

In [ ]:
from pathlib import Path
import hashlib,json,os,shutil,subprocess,sys,zipfile
REPO_URL='https://github.com/CEISCA-VT/DigitalTwinDivergence.git'; REPO_REF='main'; EXPECTED_COMMIT=''
BASE_SEEDS=[42,1042,2042]; SHARD_INDEX=0; SHARD_COUNT=6; DEVICE='cuda'
WORK=Path('/kaggle/working'); REPO=WORK/'DigitalTwinDivergence'; OUTPUT=WORK/'i2nav_v2_causal_derivative_loso'; LOGS=WORK/'i2nav_v2_causal_derivative_logs'
OUTPUT.mkdir(parents=True,exist_ok=True); LOGS.mkdir(parents=True,exist_ok=True)
if not 0 <= SHARD_INDEX < SHARD_COUNT: raise ValueError('Invalid shard')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',REPO_REF,REPO_URL,str(REPO)],check=True)
COMMIT=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if EXPECTED_COMMIT and COMMIT != EXPECTED_COMMIT: raise RuntimeError(f'Commit mismatch: {COMMIT}')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'; print(COMMIT)

In [ ]:
probe="import json,torch; print(json.dumps({'ok':torch.cuda.is_available(),'cap':list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,'arch':torch.cuda.get_arch_list() if torch.cuda.is_available() else []}))"
p=subprocess.run([sys.executable,'-c',probe],text=True,capture_output=True,check=True); gpu=json.loads(p.stdout.strip().splitlines()[-1]); print(gpu)
if not gpu['ok']: raise RuntimeError('Enable a Kaggle GPU')
if tuple(gpu['cap'])==(6,0) and 'sm_60' not in gpu['arch']:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--force-reinstall','torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu118'],check=True)
real="import torch; x=torch.randn(64,64,device='cuda'); g=torch.nn.GRU(6,64,2,batch_first=True).cuda(); y,_=g(torch.randn(2,20,6,device='cuda')); torch.cuda.synchronize(); assert torch.isfinite(x@x.T).all() and torch.isfinite(y).all()"
subprocess.run([sys.executable,'-c',real],check=True); print('CUDA PREFLIGHT: PASS')

In [ ]:
v1_runner='DigitalTwin.analysis.i2nav_loso_ablation'; v2_runner='DigitalTwin.analysis.i2nav_v2_full_loso'
v1_help=subprocess.check_output([sys.executable,'-m',v1_runner,'--help'],cwd=REPO,text=True); v2_help=subprocess.check_output([sys.executable,'-m',v2_runner,'--help'],cwd=REPO,text=True)
if '--feature-derivative-mode' not in v1_help or '--v1-checkpoint' not in v2_help: raise RuntimeError('Repository commit lacks causal V1/V2 support')
seqs=['building00','building01','building02','parking00','parking01','parking02','playground00','street00','street01','street02']
tasks=[{'test':s,'seed':seed} for s in seqs for seed in BASE_SEEDS]; tasks=sorted(tasks,key=lambda x:(x['test'],x['seed']))
shard=[t for i,t in enumerate(tasks) if i%SHARD_COUNT==SHARD_INDEX]
ledger={'schema':'causal_derivative_v2_loso_ledger_v1','commit':COMMIT,'derivative_mode':'causal_backward','total_tasks':30,'shard_index':SHARD_INDEX,'shard_count':SHARD_COUNT,'tasks':shard}
(OUTPUT/f'task_ledger_shard_{SHARD_INDEX:02d}.json').write_text(json.dumps(ledger,indent=2)+'\n')
print(f'This shard: {len(shard)} of 30 paired V1-to-V2 pipelines')

In [ ]:
env=os.environ.copy(); env['PYTHONPATH']=str(REPO)
for i,t in enumerate(shard,1):
    log=LOGS/f"test-{t['test']}__seed-{t['seed']}.log"; v1_out=OUTPUT/'v1'/f"seed_{t['seed']}"/t['test']; v2_out=OUTPUT/'v2'
    v1_cmd=[sys.executable,'-m',v1_runner,'--root',str(REPO/'public_datasets'/'im2nav'),'--output-dir',str(v1_out),'--folds',t['test'],'--methods','gru_dual','--save-trajectories','--seed',str(t['seed']),'--device',DEVICE,'--feature-derivative-mode','causal_backward']
    v1_checkpoint=v1_out/'folds'/t['test']/'gru_dual.pt'; v1_results=v1_out/'loso_results.csv'
    v2_cmd=[sys.executable,'-m',v2_runner,'--root',str(REPO/'public_datasets'/'im2nav'),'--frozen-v1-dir',str(REPO/'results'/'i2nav_v1_frozen'),'--output-dir',str(v2_out),'--test-sequence',t['test'],'--base-seed',str(t['seed']),'--device',DEVICE,'--feature-derivative-mode','causal_backward','--v1-checkpoint',str(v1_checkpoint),'--v1-results-csv',str(v1_results)]
    print(f'[{i}/{len(shard)}] {t}',flush=True)
    with log.open('a',encoding='utf-8') as f:
        if not v1_checkpoint.is_file():
            result=subprocess.run(v1_cmd,cwd=REPO,env=env,text=True,stdout=f,stderr=subprocess.STDOUT)
            if result.returncode: raise RuntimeError(f'V1 fit failed: {log}')
        result=subprocess.run(v2_cmd,cwd=REPO,env=env,text=True,stdout=f,stderr=subprocess.STDOUT)
    if result.returncode: raise RuntimeError(f'V2 fit failed: {log}')
print('SHARD EXECUTION COMPLETE')

In [ ]:
audited=[]
for t in shard:
    candidates=[]
    for p in (OUTPUT/'v2').rglob(f"*_{t['test']}/run_manifest.json"):
        m=json.loads(p.read_text())
        if m['base_seed']==t['seed']: candidates.append((p,m))
    if len(candidates)!=1: raise RuntimeError(f'Manifest count {t}: {len(candidates)}')
    p,m=candidates[0]; v1_split_path=OUTPUT/'v1'/f"seed_{t['seed']}"/t['test']/'fold_splits.json'; v1_split=json.loads(v1_split_path.read_text())[0]
    if v1_split.get('feature_derivative_mode')!='causal_backward': raise RuntimeError(f'V1 was not causal-backward: {v1_split_path}')
    expected_v1=(OUTPUT/'v1'/f"seed_{t['seed']}"/t['test']/'folds'/t['test']/'gru_dual.pt').resolve()
    if m.get('feature_derivative_mode')!='causal_backward' or Path(m.get('v1_checkpoint_source','')).resolve()!=expected_v1: raise RuntimeError(f'Wrong causal V1/V2 provenance: {p}')
    if t['test'] in set(m['training_names'])|set(m['validation_names']): raise RuntimeError(f'Test leakage: {p}')
    if not p.with_name('RUN_COMPLETE.json').is_file(): raise RuntimeError(f'Incomplete: {p}')
    audited.append({**t,'v1_split':str(v1_split_path.relative_to(OUTPUT)),'v2_manifest':str(p.relative_to(OUTPUT)),'status':'PASS'})
(OUTPUT/f'shard_audit_{SHARD_INDEX:02d}.json').write_text(json.dumps({'schema':'causal_derivative_shard_audit_v1','commit':COMMIT,'rows':audited,'complete':len(audited)==len(shard)},indent=2)+'\n')
print(f'AUDIT PASS: {len(audited)}/{len(shard)}')

In [ ]:
archive=WORK/f"i2nav_v2_causal_derivative_shard_{SHARD_INDEX:02d}_of_{SHARD_COUNT:02d}.zip"
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for root in (OUTPUT,LOGS):
        for p in root.rglob('*'):
            if p.is_file(): z.write(p,p.relative_to(WORK))
print({'archive':str(archive),'sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'bytes':archive.stat().st_size})

## Interpretation boundary
Merge all shards before comparing this model with frozen V2. Report it as a backward-derivative sensitivity. The cheaper primary check remains adding the known feature-availability delay to the original precomputed-state replay; this notebook answers the stronger optional question of what happens after retraining the model with backward-only derivative inputs.